# Neighbourhood Code Mapping Investigation

## Purpose

This notebook investigates whether `NEIGHBOURHOOD_CODE` from the City of Vancouver Property Tax Report can be translated into readable geography names using available project data and official documentation.

Notebook 08 confirmed that `NEIGHBOURHOOD_CODE` is the most suitable field for neighbourhood-level aggregation in this dataset, and Notebook 09 visualized the resulting 30-code output. However, the codes themselves are not human-readable. Before labelling any chart or output with neighbourhood names, an authoritative mapping source must be identified and validated.

This notebook does not produce any processed output files, visualizations, or final conclusions. It is an investigation-only step.

## Caveat

`NEIGHBOURHOOD_CODE` is a coded geography field. Do not treat codes as readable neighbourhood names unless an official mapping source is found.

## Imports

In [ ]:
import pandas as pd
from pathlib import Path

## Paths and Constants

All paths and configuration values are defined as named constants. `SAMPLE_SIZE` controls how many rows are loaded from the raw file to keep memory usage low during investigation. `CSV_SEPARATOR` is `";"` because the Property Tax Report uses semicolons as delimiters, confirmed in Notebook 01.

In [ ]:
RAW_PROPERTY_TAX_PATH      = Path("../data/raw/property_tax_report_raw.csv")
NEIGHBOURHOOD_OUTPUT_PATH  = Path("../data/processed/property_value_change_by_neighbourhood.csv")
CSV_SEPARATOR              = ";"
SAMPLE_SIZE                = 1000

print(f"RAW_PROPERTY_TAX_PATH     : {RAW_PROPERTY_TAX_PATH}")
print(f"NEIGHBOURHOOD_OUTPUT_PATH : {NEIGHBOURHOOD_OUTPUT_PATH}")
print(f"CSV_SEPARATOR             : {repr(CSV_SEPARATOR)}")
print(f"SAMPLE_SIZE               : {SAMPLE_SIZE:,}")

## Load Processed Neighbourhood-Code Output

We start from the already-processed output produced by Notebook 08. This gives us the full list of `neighbourhood_code` values used in the project, along with the property counts per code. Validation checks confirm the output is clean before we attempt any mapping investigation.

In [ ]:
neighbourhood_df = pd.read_csv(NEIGHBOURHOOD_OUTPUT_PATH)

print("Shape   :", neighbourhood_df.shape)
print("Columns :", neighbourhood_df.columns.tolist())
print()
print(f"Unique neighbourhood_code count  : {neighbourhood_df['neighbourhood_code'].nunique()}")
print(f"Duplicate neighbourhood_code count: {neighbourhood_df['neighbourhood_code'].duplicated().sum()}")
print(f"Missing neighbourhood_code count : {neighbourhood_df['neighbourhood_code'].isna().sum()}")
print(f"Total property_count             : {neighbourhood_df['property_count'].sum():,}")

## All Neighbourhood Codes

Display all `neighbourhood_code` values sorted ascending alongside their `property_count`. This gives a complete picture of the codes we need to map before treating any output as human-readable.

In [ ]:
codes_display = (
    neighbourhood_df[["neighbourhood_code", "property_count"]]
    .sort_values("neighbourhood_code", ascending=True)
    .reset_index(drop=True)
)

print(f"Total codes: {len(codes_display)}")
print()
print(codes_display.to_string(index=False))

## Inspect Raw Property Tax Columns

Load a 1,000-row sample from the raw Property Tax Report to inspect what other geography or name-related columns are available in the source data. We then run a keyword search over column names to identify any candidates that might carry readable neighbourhood or area labels.

In [ ]:
raw_sample = pd.read_csv(
    RAW_PROPERTY_TAX_PATH,
    sep=CSV_SEPARATOR,
    nrows=SAMPLE_SIZE,
)

print(f"Sample shape : {raw_sample.shape}")
print()
print("All columns:")
for col in raw_sample.columns:
    print(f"  {col}")

In [ ]:
GEO_KEYWORDS = [
    "neigh",
    "area",
    "local",
    "district",
    "zone",
    "geo",
    "address",
    "postal",
    "street",
    "code",
]

candidate_columns = [
    col for col in raw_sample.columns
    if any(kw in col.lower() for kw in GEO_KEYWORDS)
]

print(f"Candidate geography/name columns ({len(candidate_columns)} found):")
for col in candidate_columns:
    print(f"  {col}")

## Candidate Geography Field Summary

For each candidate column identified by the keyword search, calculate the non-null count, missing count, missing percentage, and number of unique values. Columns with lower missing rates and a moderate number of unique values are more likely to be useful as grouping or mapping fields.

In [ ]:
summary_rows = []

for col in candidate_columns:
    non_null   = int(raw_sample[col].notna().sum())
    missing    = int(raw_sample[col].isna().sum())
    unique     = int(raw_sample[col].nunique(dropna=True))
    missing_pct = round(missing / len(raw_sample) * 100, 2)
    summary_rows.append({
        "column"        : col,
        "non_null"      : non_null,
        "missing"       : missing,
        "missing_pct"   : missing_pct,
        "unique_values" : unique,
    })

candidate_summary_df = (
    pd.DataFrame(summary_rows)
    .sort_values("missing_pct")
    .reset_index(drop=True)
)

print(candidate_summary_df.to_string(index=False))

## Search for Readable Labels in Raw Data

For each candidate column, display the top 20 most frequent values including missing. This reveals whether the field contains human-readable area names, numeric codes, or structured identifiers. No assumptions are made -- value previews only.

In [ ]:
TOP_N = 20

for col in candidate_columns:
    print(f"--- {col} ---")
    vc = raw_sample[col].value_counts(dropna=False).head(TOP_N)
    print(vc.to_string())
    print()

## Official Mapping Source Investigation

An official mapping source is required before converting `neighbourhood_code` values into readable neighbourhood names. Possible sources to investigate include:

- City of Vancouver Open Data documentation for the Property Tax Report
- BC Assessment documentation or lookup tables
- City of Vancouver local area boundary data
- Any official metadata or lookup table linking `NEIGHBOURHOOD_CODE` to readable names

> **Important:** The City of Vancouver Local Area Boundary dataset contains 22 local areas, while the Property Tax Report output currently has 30 neighbourhood codes. Therefore, these should not be assumed to be equivalent without an official mapping. A 30-to-22 mismatch means a direct join would not be straightforward and any mapping must be verified against authoritative documentation before use.

_This section should be updated once an official mapping source has been located or confirmed absent._

## Preliminary Mapping Conclusion

After completing the investigation above, one of the following outcomes applies. Update this section with the result once the official mapping source has been investigated.

---

**Outcome A — Official mapping found:**
If a verified mapping table is located (e.g., from City of Vancouver Open Data or BC Assessment documentation), create a reference mapping table in a future notebook step. Document the source, access date, and any caveats. Do not apply the mapping until the source is confirmed authoritative.

**Outcome B — Official mapping not found:**
If no authoritative mapping source is identified, continue using `neighbourhood_code` as a coded geography label in all outputs. Document this limitation in the data dictionary and methodology notes. Do not substitute guessed or inferred names.

**Outcome C — Partial mapping found:**
If a partial or unofficial mapping is found (e.g., a community-maintained table or an approximate match to local area names), document the uncertainty explicitly. Do not use partial mappings as final labels in portfolio or recruiter-facing outputs until the mapping has been verified against an official source.

---

_Outcome: [To be filled in after investigation]_